# Comparing matrix vs single-job strategies for multi-OS CI coverage

This notebook walks through two common approaches to running CI tests across multiple operating systems in GitHub Actions. The goal is to understand when each strategy makes sense and what trade-offs you accept with each.

## What problem are we solving?

Most projects need to verify that their code works on more than one OS. A Python package might need to pass tests on Ubuntu, macOS, and Windows. A compiled binary might need to build on multiple Linux distros. GitHub Actions gives you two main patterns for this:

1. **Matrix strategy** — let the runner fan out the same job across multiple OS/version combinations in parallel.
2. **Single-job strategy** — write one job that handles multiple OSes through conditional steps or a single OS and rely on other signals (like cross-compilation) for coverage.

Both work. The question is which one fits your project's size, speed requirements, and maintenance budget.

## Strategy 1: Matrix approach

The matrix strategy lets you define a set of OS (and optionally version) combinations, and GitHub Actions runs the same job once for each combination. All runs happen in parallel.

In [ ]:
# .github/workflows/test-matrix.yml
name: Test (matrix)
on: [push, pull_request]

jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [ubuntu-latest, macos-latest, windows-latest]
        python-version: ['3.10', '3.11', '3.12']
      fail-fast: false
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e '.[test]'
      - run: pytest

### What this gives you

- **Parallel execution** — all 9 combinations (3 OS x 3 Python versions) run simultaneously. You get results faster.
- **Exact coverage** — each combination is tested on its native OS, so you catch OS-specific issues (file path separators, system library differences, signal handling).
- **Clear failure attribution** — when a job fails, you know exactly which OS/version combination broke.

### What it costs

- **More minutes** — 9 parallel jobs consume 9x the runner minutes. For open-source projects with free minutes, this matters.
- **More YAML** — the matrix definition grows with each dimension. Adding a fourth Python version or a second architecture multiplies the combinations.
- **Longer wall-clock if one job is slow** — the total workflow time is the slowest job, not the sum. A Windows job that takes 10 minutes blocks the whole workflow even if Ubuntu finishes in 2.

## Strategy 2: Single-job approach

Instead of fanning out, you run one job on a single OS and handle multi-OS coverage through other means: conditional steps, a separate lightweight job, or accepting that one OS is the primary target.

In [ ]:
# .github/workflows/test-single.yml
name: Test (single primary OS)
on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -e '.[test]'
      - run: pytest

  # Optional: quick smoke test on other OS
  compat:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [macos-latest, windows-latest]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -e '.[test]'
      - run: pytest --co -q  # collect tests only, verify import works

### What this gives you

- **Lower cost** — you run the full test suite once and do a lighter check on other OSes.
- **Simpler YAML** — one primary job, one optional compat job. Fewer moving parts.
- **Faster feedback** — the primary job finishes quickly; the compat job can be a quick import/smoke test.

### What it costs

- **Less coverage** — the compat job might only verify that the code imports and basic smoke tests pass. OS-specific bugs in edge cases can slip through.
- **Harder to debug** — if the compat job fails, you don't have the full test output to diagnose the issue.
- **Conditional logic complexity** — if you try to handle multiple OS in one job with `if:` conditions, the workflow becomes harder to read and maintain.

## Side-by-side comparison

| Criterion | Matrix | Single-job + compat | Single-job only |
|---|---|---|---|
| **OS coverage** | Full parallel testing on every combo | Full on primary, smoke on others | Primary OS only |
| **Runner minutes** | High (N x M combinations) | Medium (1 full + K light) | Low (1 job) |
| **Wall-clock time** | Slowest job duration | Primary job + compat job time | Primary job only |
| **Failure clarity** | Pinpoints exact OS/version | Primary failures clear; compat failures less so | Clear for primary OS |
| **YAML complexity** | Grows with matrix dimensions | Moderate | Minimal |
| **Maintenance** | Add new OS/version to matrix list | Update both jobs | Update one job |
| **Best for** | Libraries, cross-platform tools, packages | Apps with one primary OS, light cross-check | Solo projects, single-target tools |

## When to pick which

**Choose matrix when:**
- You maintain a library or package that others install on different OSes.
- OS-specific behavior is a real concern (file paths, system calls, compiled extensions).
- You have enough CI minutes budget (or the project is open-source with free minutes).
- You want clear failure attribution for every combination.

**Choose single-job + compat when:**
- Your app runs primarily on one OS (e.g., a server deployed to Ubuntu) but you want to catch obvious portability issues.
- CI budget is tight and you can't afford full parallel testing.
- You're willing to accept lighter coverage on non-primary OSes.

**Choose single-job only when:**
- The project is a solo tool or script that runs on one OS.
- Cross-platform support is not a goal.
- Speed and simplicity matter more than coverage.

## Common gotchas

1. **`fail-fast: false`** — By default, matrix jobs stop other runs when one fails. For multi-OS testing, you usually want `fail-fast: false` so you see all failures at once.

2. **Runner label drift** — `ubuntu-latest`, `macos-latest`, and `windows-latest` point to different versions over time. Pin to a specific version (e.g., `ubuntu-22.04`) if reproducibility matters more than staying current.

3. **Artifacts and caching** — Matrix jobs are independent. If you build artifacts in one job and need them in another, you'll need `actions/upload-artifact` and `actions/download-artifact`. Cache keys should include the matrix OS to avoid cross-OS cache pollution.

4. **Conditional steps** — If you use `if: runner.os == 'Windows'` inside a matrix job, you're mixing strategies. This works but makes the job harder to reason about. Consider splitting into separate jobs instead.

## What I'd try next

- Experiment with `fail-fast` behavior in a real project to see how it affects debugging workflow.
- Try a hybrid approach: full matrix for unit tests, single-job for integration tests that need specific OS setup.
- Look into reusable workflows to DRY up the matrix definition across multiple workflow files.